In [17]:
import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import lightgbm as lgb

In [2]:
X_train_1 =pd.read_csv("../data/boost/X_train_boost_part1.csv", index_col = "index")
X_train_2 =pd.read_csv("../data/boost/X_train_boost_part2.csv", index_col = "index")
X_train_3 =pd.read_csv("../data/boost/X_train_boost_part3.csv", index_col = "index")
X_train = pd.concat([X_train_1,X_train_2,X_train_3])

X_test = pd.read_csv("../data/boost/X_test_boost.csv", index_col = "index")

y_train = pd.read_csv("../data/y_train.csv", index_col = "index")
y_test = pd.read_csv("../data/y_test.csv", index_col = "index")


In [ ]:
# changing categorical variables into correct type
categorical_cols = ['flat_type', 'town', 'flat_model', 'block', 'street_name' ] 

for col in categorical_cols:
    X_train[col] = X_train[col].astype('category')
    X_test[col] = X_test[col].astype('category')

In [ ]:
# handle date as it is not accepted by lightgbm
def add_fractional_year(df, date_col='month', new_col='fractional_year'):
    df[new_col] = (
        df[date_col].dt.year +
        (df[date_col].dt.month - 1) / 12 +
        (df[date_col].dt.day - 1) / 365
    ).round(5)  # Optional: rounding for precision
    return df

# try using year only first
X_train_light = add_fractional_year(X_train, date_col='month', new_col='month_fraction')
X_train_light.drop(columns='month', inplace=True) 

X_test_light = add_fractional_year(X_test, date_col='month', new_col='month_fraction')
X_test_light.drop(columns='month', inplace=True) 

In [ ]:
# transforming dataset into LGB optimised version
lgb_train = lgb.Dataset(X_train_light, 
                        label=y_train.values.ravel(), 
                        categorical_feature=categorical_cols if categorical_cols else 'auto')

In [15]:
# setting parameters
params = {
    'objective': 'regression',           # Type of task: regression
    'metric': 'rmse',                    # Root Mean Squared Error
    'boosting_type': 'gbdt',             # Gradient Boosting Decision Trees
    'learning_rate': 0.1,                # Step size shrinkage
    'num_leaves': 31,                    # Max leaf nodes per tree
    'max_depth': -1,                     # No limit (-1)
    'feature_fraction': 0.9,             # Randomly select 90% of features for each tree
    'bagging_fraction': 0.8,             # Randomly select 80% of data for each iteration
    'bagging_freq': 5,                   # Perform bagging every 5 iterations
    'verbose': 1                      # Suppress logs
}

In [16]:
from sklearn.model_selection import train_test_split

X_train_sub, X_valid, y_train_sub, y_valid = train_test_split(
    X_train_light, y_train, test_size=0.2, random_state=42
)

lgb_train = lgb.Dataset(X_train_sub, label=y_train_sub.values.ravel(), categorical_feature=categorical_cols if categorical_cols else 'auto')
lgb_valid = lgb.Dataset(X_valid, label=y_valid.values.ravel(), reference=lgb_train)

model = lgb.train(
    params,
    lgb_train,
    valid_sets=[lgb_train, lgb_valid],
    valid_names=['train', 'valid'],
    num_boost_round=1000
)



[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.017936 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3539
[LightGBM] [Info] Number of data points in the train set: 597519, number of used features: 8
[LightGBM] [Info] Start training from score 323716.792710


In [ ]:
prediction = model.predict()